# Iterators (merge two sorted iterators in O(1) space)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Heaps, Two Pointers · **Difficulty/Frequency:** Popular! (10/10)

## Concepts

**What this problem is really testing:**
- What an **iterator** actually is, and why you cannot look at its next value without consuming it
- The **k-way merge** pattern — the merge half of merge sort, applied to streams instead of arrays
- **Min-heap (priority queue)** as the generalisation from 2 streams to n streams

**First-principles primer — what is each piece?**

- **Iterator** — an object with one capability: "give me the next item" (`next(it)` in Python), raising `StopIteration` when there are none left. It is *lazy*: nothing is computed or loaded until you ask. That is why it can represent a 500 GB file or an infinite sequence in a few bytes of memory.
- **Why you can't "peek"** — the iterator protocol has no `peek()`. Asking for the next value *consumes* it. So if you want to know what is coming without losing it, **you** must pull it out and hold it in a variable. That held value is called the **head** (or a *one-element buffer*).
- **Min-heap (priority queue)** — a binary tree kept in an array where every parent is `<=` its children. That single rule means the smallest element is always at index 0 (`O(1)` to look at), while inserting or removing costs `O(log k)`. Python's `heapq` gives you `heappush` / `heappop` over a plain list.
- **Two pointers** — with two sorted inputs, keeping one "cursor" into each and always advancing the one pointing at the smaller value is the classic merge. Here the cursors *are* the two held heads.

**The key insight:**

Both iterators are already sorted. So the smallest value you have not yet returned must be at the **front** of one of them — it cannot be hiding deeper inside, because everything deeper is larger. You therefore never need more than **one buffered value per iterator**, no matter how many billions of elements they hold. That is where O(1) space comes from.

**Simple worked example.** `A = [1, 3, 5]`, `B = [2, 3, 8]`. Hold one head from each and repeatedly emit the smaller:

| step | head of A | head of B | emit | why |
|---|---|---|---|---|
| 1 | **1** | 2 | `1` | 1 <= 2, advance A |
| 2 | **3** | 2 | `2` | 2 < 3, advance B |
| 3 | **3** | 3 | `3` | tie -> `<=` takes A, advance A |
| 4 | 5 | **3** | `3` | 3 < 5, advance B |
| 5 | **5** | 8 | `5` | 5 <= 8, advance A |
| 6 | *empty* | **8** | `8` | A exhausted, drain B |

Output: `1, 2, 3, 3, 5, 8` — sorted, using two variables.

**Scaling to n iterators:** step 1 of every round is "which of the heads is smallest?". With 2 heads that's one comparison; with n heads it's n-1 comparisons *per element*. A min-heap answers the same question in O(log n) by keeping the heads partially ordered, so you never re-compare everything from scratch.

## Problem Statement

Given **two iterators whose values are in strictly increasing order**, implement:

| Method | Behaviour |
|---|---|
| `hasNext()` | `True` if any value remains across both iterators |
| `next()` | The smallest not-yet-returned value across both |

**Constraint:** constant extra space — you may **not** drain the iterators into a list.

**Follow-up:** handle `n` iterators instead of 2.

**Example**

```python
m = MergedIterator(iter([1, 3, 5]), iter([2, 3, 8]))
out = []
while m.hasNext():
    out.append(m.next())
# out == [1, 2, 3, 3, 5, 8]
```

### Approach 1 — Naive (drain both, then sort)

**Idea:** pull everything out of both iterators into one list, sort it, and hand the values back one at a time.

It is correct, and it is what most people write first — but it violates the stated constraint in two ways: it buffers **every** element, and it does all its work eagerly in the constructor, so it can never be used on an infinite or very large stream.

**Time complexity:** O((m+n) log(m+n)) up front to sort; O(1) per `next()` afterwards.

**Space complexity:** **O(m+n)** — the whole merged list is materialised. This is the part the problem forbids.

In [ ]:
from typing import Any, Iterator, List, Optional


class DrainAndSortIterator:
    """Baseline: correct, but buffers everything - fails the O(1) space requirement."""

    def __init__(self, it1: Iterator, it2: Iterator) -> None:
        self.buf: List[Any] = sorted(list(it1) + list(it2))   # O(m+n) space, eager
        self.i = 0

    def hasNext(self) -> bool:
        return self.i < len(self.buf)

    def next(self) -> Any:
        if not self.hasNext():
            raise StopIteration
        val = self.buf[self.i]
        self.i += 1
        return val

### Approach 2 — Optimal for two iterators (hold one head each)

**Idea:** keep exactly one buffered value per iterator — its **head** — plus a boolean saying whether that iterator is still alive. The constructor pulls the first value from each (you have to, since iterators cannot be peeked).

Each `next()`:
1. If both are alive, return the smaller head and advance **only** that iterator.
2. If one is exhausted, return the other's head and advance it — no comparison needed.

**Why `<=` and not `<` on the tie:** when both heads are equal, `<=` takes from iterator 1 first. Either choice returns the same multiset, but fixing the rule makes the behaviour deterministic and keeps the merge *stable* (equal values come out in iterator order), which matters when the values carry payloads.

**Time complexity:** **O(1)** per `next()` and `hasNext()` — one comparison, one advance.

**Space complexity:** **O(1)** — two values and two flags, regardless of stream length.

In [ ]:
class MergedIterator:
    """Merge two sorted iterators lazily, in constant space."""

    def __init__(self, it1: Iterator, it2: Iterator) -> None:
        self.it1, self.it2 = it1, it2
        self.v1: Optional[Any] = None
        self.v2: Optional[Any] = None
        self.has1 = True
        self.has2 = True
        self._advance1()          # iterators can't be peeked, so buffer one value each
        self._advance2()

    def _advance1(self) -> None:
        try:
            self.v1 = next(self.it1)
        except StopIteration:
            self.has1, self.v1 = False, None      # flag, don't crash

    def _advance2(self) -> None:
        try:
            self.v2 = next(self.it2)
        except StopIteration:
            self.has2, self.v2 = False, None

    def hasNext(self) -> bool:
        return self.has1 or self.has2

    def next(self) -> Any:
        if not self.has1 and not self.has2:
            raise StopIteration

        if self.has1 and self.has2:
            if self.v1 <= self.v2:                # `<=` => ties favour iterator 1 (stable)
                result = self.v1
                self._advance1()
            else:
                result = self.v2
                self._advance2()
        elif self.has1:                           # it2 is drained - just pour out it1
            result = self.v1
            self._advance1()
        else:
            result = self.v2
            self._advance2()
        return result

### Follow-up — n iterators — Approach 1 — Naive (scan all n heads every time)

**Idea:** the same "hold one head each" trick, but with a list of heads. To find the smallest, compare all of them.

This is a perfectly reasonable first answer and keeps space at O(n). The problem is that finding the minimum is a **linear scan repeated for every single output element**.

**Time complexity:** **O(n) per `next()`** → O(T × n) to drain T total elements.

**Space complexity:** O(n) — one head per iterator.

In [ ]:
class MergedIteratorsScan:
    """n-way merge by scanning every head on each next(). O(n) per call."""

    def __init__(self, iterators: List[Iterator]) -> None:
        self.iterators = list(iterators)
        self.heads: List[Optional[Any]] = []
        self.alive: List[bool] = []
        for it in self.iterators:
            try:
                self.heads.append(next(it))
                self.alive.append(True)
            except StopIteration:
                self.heads.append(None)
                self.alive.append(False)

    def hasNext(self) -> bool:
        return any(self.alive)

    def next(self) -> Any:
        best = -1
        for i, ok in enumerate(self.alive):       # O(n) linear scan, every single call
            if ok and (best == -1 or self.heads[i] < self.heads[best]):
                best = i
        if best == -1:
            raise StopIteration
        result = self.heads[best]
        try:
            self.heads[best] = next(self.iterators[best])
        except StopIteration:
            self.alive[best] = False
            self.heads[best] = None
        return result

### Follow-up — n iterators — Approach 2 — Optimal (min-heap of heads)

**Idea:** the repeated question is always *"which head is smallest?"*. A min-heap answers exactly that, and — crucially — **remembers** the partial ordering between calls instead of rediscovering it.

Seed the heap with `(first_value, iterator_index)` for every non-empty iterator. Each `next()`:
1. `heappop` the smallest pair — that is the answer,
2. pull the next value from **that same** iterator and `heappush` it back (if any).

The heap therefore holds **at most one entry per iterator** at all times, which is what bounds space at O(n) rather than O(total elements).

**Why the index is in the tuple:** it tells you which iterator to refill after popping. It also breaks ties between equal values deterministically — and, importantly, avoids Python trying to compare the payloads themselves when values tie.

**Time complexity:** **O(log n) per `next()`**; O(n log n) to build. Draining T elements costs O(T log n) versus O(T × n) for the scan.

**Space complexity:** O(n) — the heap never exceeds one entry per iterator.

In [ ]:
import heapq


class MergedIterators:
    """n-way merge via a min-heap of (value, iterator index). O(log n) per next()."""

    def __init__(self, iterators: List[Iterator]) -> None:
        self.iterators = list(iterators)
        self.heap: List[Any] = []
        for i, it in enumerate(self.iterators):
            try:
                heapq.heappush(self.heap, (next(it), i))   # seed with one value per iterator
            except StopIteration:
                pass                                       # empty iterator: nothing to seed

    def hasNext(self) -> bool:
        return len(self.heap) > 0                          # heap empty <=> everything drained

    def next(self) -> Any:
        if not self.heap:
            raise StopIteration
        val, idx = heapq.heappop(self.heap)                # O(log n): smallest head
        try:
            heapq.heappush(self.heap, (next(self.iterators[idx]), idx))   # refill from the SAME iterator
        except StopIteration:
            pass                                           # that iterator is finished
        return val

## Verification

Check the worked example, then confirm all four implementations produce the identical merged sequence on randomised inputs, plus every documented edge case.

In [ ]:
import random


def drain(merged) -> List[Any]:
    out = []
    while merged.hasNext():
        out.append(merged.next())
    return out


# --- The worked example ---
assert drain(MergedIterator(iter([1, 3, 5]), iter([2, 3, 8]))) == [1, 2, 3, 3, 5, 8]
assert drain(DrainAndSortIterator(iter([1, 3, 5]), iter([2, 3, 8]))) == [1, 2, 3, 3, 5, 8]
assert drain(MergedIterators([iter([1, 3, 5]), iter([2, 3, 8])])) == [1, 2, 3, 3, 5, 8]
assert drain(MergedIteratorsScan([iter([1, 3, 5]), iter([2, 3, 8])])) == [1, 2, 3, 3, 5, 8]

# --- Edge cases ---
assert drain(MergedIterator(iter([]), iter([]))) == []              # both empty
assert drain(MergedIterator(iter([1, 2]), iter([]))) == [1, 2]      # one empty
assert drain(MergedIterator(iter([]), iter([1, 2]))) == [1, 2]
assert drain(MergedIterator(iter([1, 2, 3]), iter([9]))) == [1, 2, 3, 9]   # very unequal lengths
assert drain(MergedIterators([])) == []                             # zero iterators
assert drain(MergedIterators([iter([]), iter([])])) == []           # all iterators empty
assert drain(MergedIterators([iter([7])])) == [7]                   # a single iterator

# next() past the end must raise, not return garbage
m = MergedIterator(iter([1]), iter([]))
assert m.next() == 1
assert not m.hasNext()
try:
    m.next()
except StopIteration:
    pass
else:
    raise AssertionError("next() past the end must raise StopIteration")

# Ties across iterators: `<=` keeps iterator 1 first (stable merge)
assert drain(MergedIterator(iter([1, 1]), iter([1]))) == [1, 1, 1]

# Laziness: nothing beyond the buffered head is consumed before it is asked for
consumed = []


def spy(values):
    for v in values:
        consumed.append(v)
        yield v


m = MergedIterator(spy([10, 20, 30]), iter([1, 2]))
assert consumed == [10], "constructor must buffer exactly one value per iterator"
m.next(); m.next()                                   # both come from the second iterator
assert consumed == [10], "the larger stream must not be advanced yet"

# --- All four agree on randomised sorted inputs ---
random.seed(11)
for _ in range(200):
    a = sorted(random.sample(range(200), random.randint(0, 12)))
    b = sorted(random.sample(range(200), random.randint(0, 12)))
    expected = sorted(a + b)
    assert drain(MergedIterator(iter(a), iter(b))) == expected
    assert drain(DrainAndSortIterator(iter(a), iter(b))) == expected
    assert drain(MergedIterators([iter(a), iter(b)])) == expected
    assert drain(MergedIteratorsScan([iter(a), iter(b)])) == expected

# --- n-way agreement, n from 1 to 8, with duplicates allowed across iterators ---
for _ in range(100):
    n = random.randint(1, 8)
    lists = [sorted(random.choices(range(50), k=random.randint(0, 10))) for _ in range(n)]
    expected = sorted(x for lst in lists for x in lst)
    assert drain(MergedIterators([iter(l) for l in lists])) == expected
    assert drain(MergedIteratorsScan([iter(l) for l in lists])) == expected

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Sorted but not *strictly* sorted (duplicates inside one iterator).** Nothing changes. The invariant is "each head is the smallest unreturned value of its own iterator", which never assumed distinctness. `<=` already resolves ties, and the randomised tests above deliberately feed duplicates through `random.choices`.
- **Very large k.** Compare the two costs directly: the scan is O(T × k) to drain T elements, the heap is O(T log k). The heap wins as soon as k is more than a handful. If k is enormous but *most* iterators are empty or short, note that the heap self-shrinks — an exhausted iterator is simply never pushed back, so `log k` decays toward `log(live k)`.
- **One iterator is infinite.** Both designs still work, because both are fully lazy: no value is pulled until it is needed. `hasNext()` will simply never return `False`, so the caller must stop on its own condition (a value threshold, a count). The naive drain-and-sort approach hangs forever — a sharp illustration of why laziness was required.
- **Generator instead of a class.** `heapq.merge(*iterators)` is exactly this, in the standard library, and a hand-rolled `def merge(a, b): ... yield ...` is a few lines. What you lose is an explicit `hasNext()`: generators only tell you they are finished by raising `StopIteration`, so a caller who needs to *ask* before consuming has to wrap it back up in a class with a buffered head — the same one-element buffer as Approach 2.
- **Custom comparison keys.** Push `(key(value), tiebreak_index, value)` so the heap orders by the key, and the index prevents Python from ever comparing two `value` payloads when the keys are equal (which would raise `TypeError` for non-comparable objects). For the two-iterator version, store the key alongside each head and compare keys instead of values.

## Empirical complexity check

Compare the n-way **linear scan** (O(n) per element) against the n-way **min-heap** (O(log n) per element), draining a fixed total number of elements while the **number of iterators** doubles.

| Growth when the iterator count doubles | What it means |
|---|---|
| ~2x | linear per element — the scan re-examines every head each time |
| ~1x | logarithmic — the heap barely notices twice as many streams |

Total elements is held constant so that only the per-element *selection* cost varies.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

TOTAL = 40000          # total elements is FIXED; only the number of iterators grows


def make_iterator_lists(num_iters):
    per = max(1, TOTAL // num_iters)
    # Interleaved arithmetic progressions: each list is sorted, and they overlap heavily.
    return ([[i * num_iters + k for i in range(per)] for k in range(num_iters)],)


def run_scan(lists):
    m = MergedIteratorsScan([iter(l) for l in lists])
    while m.hasNext():
        m.next()


def run_heap(lists):
    m = MergedIterators([iter(l) for l in lists])
    while m.hasNext():
        m.next()


benchmark(
    {"n-way naive - scan all heads O(n)": run_scan,
     "n-way optimal - min-heap O(log n)": run_heap},
    make_iterator_lists,
    sizes=[8, 16, 32, 64, 128],
    repeats=2,
)

## Patterns learned

- **Sorted inputs mean the answer is always at a front.** If every input is ordered, the global minimum is one of the k heads — never buried. That single fact is what licenses O(1)/O(k) space instead of O(total).
- **One-element buffer = peek.** Any "I need to look ahead without consuming" problem on a lazy stream is solved by pulling one value out and holding it. The buffered head *is* your `peek()`.
- **A repeated "who is smallest?" question wants a heap.** Whenever a loop re-scans the same candidate set for a min/max, a priority queue turns O(k) per round into O(log k) by *retaining* the ordering work between rounds. Same pattern as Dijkstra, task schedulers, and top-K.
- **Put a tiebreaker in your heap tuples.** `(value, index, payload)` — the index makes ordering deterministic *and* stops Python from comparing payloads that may not support `<`.
- **Flags beat sentinel values for "stream finished".** A separate `alive` boolean is unambiguous; a magic sentinel like `None` or `float('inf')` collides with real data and quietly corrupts comparisons.
- **Stay lazy when the constraint says constant space.** Doing work only in `next()` is what lets the same code serve a 3-element list and an infinite generator.